# Orchestador Principal - Modelos de Recomendación Anime

Este notebook se encarga de unificar el proceso de carga de datos, instanciación y comparación de todas las arquitecturas de recomendación (KNN, PMF, BMF, NCF).

In [2]:
import polars as pl
import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from collections import defaultdict
import matplotlib.pyplot as plt   
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Configurar estilo visual para las gráficas
sns.set_theme(style="whitegrid")

## 1. Carga de Datos y Mapeos

In [3]:
# Cargar DataFrames en Pandas (KNN usa set y diccionarios de python que iteran mas comodo sobre pd o tuples)
df_train_pl = pl.read_parquet("data/train.parquet")
df_test_pl = pl.read_parquet("data/test.parquet")

df_train = df_train_pl.to_pandas()
df_test = df_test_pl.to_pandas()

# Cargar los mapeos guardados por preprocess.py
with open("data/mapeos.pkl", "rb") as f:
    mapeos = pickle.load(f)

user2idx = mapeos['user2idx']
anime2idx = mapeos['anime2idx']

NUM_USERS = len(user2idx)
NUM_ITEMS = len(anime2idx)

print(f"Usuarios únicos: {NUM_USERS}")
print(f"Items únicos: {NUM_ITEMS}")
print(f"Interacciones Train: {len(df_train):,}")
print(f"Interacciones Test: {len(df_test):,}")

Usuarios únicos: 47143
Items únicos: 6532
Interacciones Train: 4,915,940
Interacciones Test: 1,228,986


## 2. Preparación para PMF (Matrices Dispersas)

In [4]:
# PMF necesita las matrices dispersas CSR generadas a partir de ratings (1 a 10)
R_train_sparse = sp.csr_matrix(
    (df_train['rating'].values, (df_train['user_id'].values, df_train['anime_id'].values)), 
    shape=(NUM_USERS, NUM_ITEMS)
)

R_test_sparse = sp.csr_matrix(
    (df_test['rating'].values, (df_test['user_id'].values, df_test['anime_id'].values)), 
    shape=(NUM_USERS, NUM_ITEMS)
)

MU = df_train['rating'].mean()
print(f"Media global de entrenamiento (MU): {MU:.4f}")

Media global de entrenamiento (MU): 7.7929


## 3. Modelo 1 - KNN (K-Nearest Neighbors)

### Guía Teórica y Técnica: Modelo KNN para el Sistema de Recomendación de Anime

Este documento proporciona una explicación detallada del modelo **K-Nearest Neighbors (KNN)** implementado en el módulo `knn.py`, sirviendo como preámbulo teórico y técnico antes de ejecutar el análisis y optimización en `main.ipynb`. 

---

#### 1. Introducción a KNN en Sistemas de Recomendación

El algoritmo **K-Nearest Neighbors (K-Vecinos Más Cercanos)** es un método no paramétrico clásico dentro de los sistemas de recomendación basados en **Filtrado Colaborativo**. Su premisa fundamental es que las preferencias pasadas de los usuarios estables reflejan comportamientos futuros similares.

En tu arquitectura coexisten **dos variantes complementarias** de este enfoque:

1. **User-Based Collaborative Filtering (Basado en Usuarios):** Es el corazón de la clase personalizada `KNNRecommender`. Para predecir el interés de un usuario activo $u$ por un anime $i$, el algoritmo identifica a los $K$ usuarios más similares a $u$ que ya han calificado al anime $i$, combinando sus desviaciones y preferencias.
2. **Item-Based Collaborative Filtering (Basado en Ítems):** Es la variante ejecutada a través de `scikit-learn` en la función `generar_resultados_knn_frontend`. Calcula la similitud geométrica entre vectores de animes en el espacio de calificaciones de los usuarios. Es más estable ante catálogos con interacciones masivas y es ideal para construir el grafo de relaciones (*Source-Target*) del frontend.

---

#### 2. Preparación y Estructura Eficiente de Datos

Un desafío crítico en los modelos KNN es la **dispersión (sparsity)** de la matriz de utilidad Usuario-Ítem. En sistemas reales, un usuario común califica un porcentaje ínfimo del catálogo completo de animes. Si se representara esta información como una matriz densa tradicional, los millones de celdas vacías (`NaN` o `0`) colapsarían la memoria RAM.

Tu código aborda este desafío de dos formas óptimas:
* **Estructuras de Diccionarios Indexados (`ratings_train`):** En `KNNRecommender`, se construyen mapas hash de Python (`dict` de `dict`) que indexan únicamente las interacciones existentes. Esto reduce el costo computacional de buscar elementos de $\mathcal{O}(N)$ a $\mathcal{O}(1)$ en promedio.
* **Matrices CSR (Compressed Sparse Row):** En la integración con `sklearn`, se utiliza `scipy.sparse.csr_matrix`. Esta estructura almacena en vectores contiguos los valores indexados por filas, eliminando los ceros del cómputo y permitiendo operaciones vectoriales ultrarrápidas mediante álgebra lineal en C/C++.

---

#### 3. Métricas de Similitud Implementadas

El cálculo de la "cercanía" geométrica define la precisión de los vecinos elegidos. Tu archivo `knn.py` implementa tres métricas fundamentales con propiedades matemáticas particulares:

##### A. Similitud de Correlación (Pearson)
Mide la relación lineal entre las calificaciones co-emitidas por dos usuarios $u$ y $v$. Su gran ventaja matemática es que **corrige el sesgo de calificación individual** (el centrado respecto a la media de cada usuario).

$$
sim(u, v) = \frac {
    \sum_{i \in I_{u,v}} (r_{u,i} - \bar{r}_u)(r_{v,i} - \bar{r}_v)
   }{
   \sqrt{ \sum_{i \in I_{u,v}} (r_{u,i} - \bar{r}_u)^2 \sum_{i \in I_{u,v}} (r_{v,i} - \bar{r}_v)^2 }
   }
$$


##### B. Similitud JMSD (Jaccard + Mean Squared Difference)
Es una métrica híbrida de última generación para sistemas de recomendación que mitiga las limitaciones individuales de Jaccard y MSD:

1. **Coeficiente de Jaccard:** Evalúa la proporción de solapamiento del consumo implícito, midiendo cuántos animes comparten en comparación con la unión de sus catálogos visitados.
2. **Mean Squared Difference (MSD):** Evalúa la distancia euclidiana promedio al cuadrado de las calificaciones sobre los ítems comunes.

$$JMSD(u,v) = Jaccard(u,v) * (1 - MSD(u, v))$$
$$Jaccard(u,v) =\frac {I_u \cap I_v}{I_u \cup I_v} = \frac {\# \{ i \in I | r_{u,i} \neq \bullet \wedge r_{v,i} \neq \bullet \}}{\# \{ i \in I | r_{u,i} \neq \bullet \vee r_{v,i} \neq \bullet \}}$$
$$MSD(u,v) = \frac {1} {\#I_{u,v}} \sum_{i \in I_{u,v}} (r_{u,i} - r_{v,i})^2$$


*Nota técnica:* JMSD destaca porque equilibra la similitud cuantitativa (coincidencia exacta en la nota) con la similitud cualitativa (coincidencia en el hábito de ver el mismo anime).

---

#### 4. El Algoritmo de Predicción (Fórmula de Resnick)

Una vez seleccionados los $K$ vecinos más cercanos al usuario $u$ que han consumido el anime $i$, no basta con promediar sus notas de forma simple. Los usuarios tienen diferentes escalas internas (un usuario exigente puede considerar un $7/10$ una obra maestra, mientras que otro optimista califica todo con $9/10$).

Nuestra función `prediction_knn` implementa la **Fórmula de Resnick con ajuste por la media del usuario**, la cual normaliza el comportamiento y calcula una suma ponderada por la similitud del vecino:

$$\hat{r}_{u,i} = \bar{r}_{u} + \frac{\sum_{v \in N} sim(u,v) \cdot (r_{v,i} - \bar{r}_v)}{\sum_{v \in N} |sim(u,v)|} $$

Donde:
* $\hat{r}_{u,i}$ es la calificación predicha para el usuario $u$ sobre el anime $i$.
* $ \bar{r}_{u}$ es la media global del usuario activo (la línea base sobre la cual se proyecta la recomendación).
* $r_{v,i} -  \bar{r}_{u}$ es la desviación del vecino $v$; indica si al vecino ese anime le gustó más o menos de lo que le suele gustar el anime promedio.
* El denominador normaliza la influencia asegurando que el resultado permanezca estrictamente en la escala permitida (restringida mediante `np.clip` entre `R_MIN=1` y `R_MAX=10`).

---

#### 5. Protocolo de Evaluación y Métricas de Rendimiento

Para validar empíricamente el comportamiento del modelo según la variación de $K$, la función `evaluate` implementa tres indicadores clave:

1. **RMSE (Root Mean Squared Error):** Penaliza severamente los errores grandes debido a la elevación al cuadrado. Es el estándar de la industria.

2. **MAE (Mean Absolute Error):** Mide la magnitud promedio del error en valores absolutos, ofreciendo una interpretación lineal directa del desvío (ej. "el modelo se equivoca en promedio $\pm 1.1$ puntos").
 
3. **Cobertura (Prediction Coverage):** Determina el porcentaje de interacciones del conjunto de prueba para las cuales el modelo fue capaz de encontrar vecinos válidos y generar una predicción numérica real. Si un anime es extremadamente raro, o un usuario no comparte historial con nadie, la predicción retorna `None` y reduce la cobertura.



**OUTPUT ESPERADO**

1. **La Gráfica del Codo**

2. **Evaluación Transparente**

3. **Muestra Inteligente**: El DataFrame de pandas final creará una nueva columna llamada Diferencia_Absoluta pintada en un gradiente rojo. Esto  permitirá ver de un vistazo en qué usuarios el KNN acertó (colores claros o blancos) y en cuáles falló por mucha diferencia de nota (colores rojos intensos).

In [4]:
from knn import run_knn

# 1. Ejecutamos el modelo obteniendo 4 variables de retorno
# Nota: Puedes cambiar force_recompute=True si cambias los k_values y quieres re-entrenar
df_results_knn, knn_model, best_k, best_df_preds = run_knn(
    df_train, 
    df_test.sample(1500, random_state=42), 
    k_values=[5, 10, 20, 30, 50, 75, 100],
    force_recompute=False 
)

# 2. Gráfica de Elección del K Óptimo
plt.figure(figsize=(10, 5))
plt.plot(df_results_knn['K'], df_results_knn['RMSE'], marker='o', color='#1f77b4', label='RMSE (Error Cuadrático)')
plt.plot(df_results_knn['K'], df_results_knn['MAE'], marker='s', color='#ff7f0e', label='MAE (Error Absoluto)')

# Extraer el mejor valor para resaltarlo visualmente
best_rmse = df_results_knn['RMSE'].min()
plt.axvline(x=best_k, color='red', linestyle='--', alpha=0.6, label=f'K Óptimo ({best_k})')
plt.scatter(best_k, best_rmse, color='red', s=100, zorder=5) # Punto rojo en el mínimo

plt.title('Evolución del Error según el Número de Vecinos (K)', fontsize=14)
plt.xlabel('Número de Vecinos (K)', fontsize=12)
plt.ylabel('Métricas de Error', fontsize=12)
plt.xticks(df_results_knn['K']) # Mostrar todos los valores de K en el eje X
plt.legend()
plt.show()

# 3. Impresión de Métricas de Evaluación
best_mae = df_results_knn.loc[df_results_knn['K'] == best_k, 'MAE'].values[0]
best_cob = df_results_knn.loc[df_results_knn['K'] == best_k, 'Cobertura'].values[0]

print("="*40)
print(f"🏆 MÉTRICAS DEL MEJOR MODELO KNN (K={best_k}) 🏆")
print("="*40)
print(f"RMSE (Root Mean Square Error): {best_rmse:.4f}")
print(f"MAE (Mean Absolute Error)    : {best_mae:.4f}")
print(f"Cobertura de Predicción      : {best_cob:.2f}%")
print("="*40)

# 4. Muestra de predicciones en X usuarios
print("\n👀 Muestra de Predicciones Realizadas vs Rating Real:")
# Tomamos 10 usuarios/items aleatorios de la evaluación para comparar
muestra = best_df_preds.sample(10, random_state=42)

# Coloreamos las columnas para ver la diferencia entre lo predicho y lo real
muestra['Diferencia_Absoluta'] = abs(muestra['rating_real'] - muestra['rating_predicho'])
display(muestra.sort_values(by='Diferencia_Absoluta').reset_index(drop=True).style.background_gradient(subset=['Diferencia_Absoluta'], cmap='Reds'))


>> Inicializando modelo KNN...
Construyendo matriz interna KNN...
Cargando resultados de K guardados previamente desde results/resultados_k_optimo.csv...
>> Generando predicciones de muestra para el mejor K (k = 10) a partir del test...


KeyboardInterrupt: 

## 4. Modelo 2 - PMF (Probabilistic Matrix Factorization)

La función `run_pmf` se llama desde el notebook principal para entrenar y evaluar el modelo PMF. Esta función crea el modelo, lo entrena con `R_train_sparse`, lo evalúa con `R_test_sparse` y guarda los resultados obtenidos. El modelo PMF predice una valoración usando la media global, el sesgo del usuario, el sesgo del anime y el producto de los factores latentes de usuario y anime.

La llamada devuelve cuatro elementos: `df_results_pmf`, que contiene el RMSE por época; `pmf_model`, que es el modelo entrenado; `pmf_best_rmse`, que guarda el mejor RMSE obtenido en test; y `df_preds_pmf`, que contiene las valoraciones reales y predichas por el modelo.

In [5]:
from pmf import run_pmf

df_results_pmf, pmf_model, pmf_best_rmse, df_preds_pmf = run_pmf(
    R_train_sparse,
    R_test_sparse,
    mu=MU,
    n_users=NUM_USERS,
    n_items=NUM_ITEMS,
    n_factors=50,
    lr=0.005,
    reg=0.05,
    epochs=30,
    patience=5,
    force_recompute=True
)

print(f"El mejor RMSE logrado por PMF fue: {pmf_best_rmse:.4f}")

>> Inicializando modelo PMF...
>> Entrenando modelo PMF...
  Época  1/30 | Train RMSE: 1.2609 | Test RMSE: 1.2701
  Época  2/30 | Train RMSE: 1.2200 | Test RMSE: 1.2330
  Época  3/30 | Train RMSE: 1.2012 | Test RMSE: 1.2169
  Época  4/30 | Train RMSE: 1.1866 | Test RMSE: 1.2052
  Época  5/30 | Train RMSE: 1.1709 | Test RMSE: 1.1930
  Época  6/30 | Train RMSE: 1.1562 | Test RMSE: 1.1822
  Época  7/30 | Train RMSE: 1.1430 | Test RMSE: 1.1731
  Época  8/30 | Train RMSE: 1.1311 | Test RMSE: 1.1655
  Época  9/30 | Train RMSE: 1.1194 | Test RMSE: 1.1588
  Época 10/30 | Train RMSE: 1.1076 | Test RMSE: 1.1525
  Época 11/30 | Train RMSE: 1.0956 | Test RMSE: 1.1465
  Época 12/30 | Train RMSE: 1.0841 | Test RMSE: 1.1413
  Época 13/30 | Train RMSE: 1.0721 | Test RMSE: 1.1360
  Época 14/30 | Train RMSE: 1.0605 | Test RMSE: 1.1315
  Época 15/30 | Train RMSE: 1.0493 | Test RMSE: 1.1274
  Época 16/30 | Train RMSE: 1.0385 | Test RMSE: 1.1238
  Época 17/30 | Train RMSE: 1.0281 | Test RMSE: 1.1204
  Époc

| Modelo | n_factors | lr | reg | epochs | patience | best_RMSE |
|---|---:|---:|---:|---:|---:|---:|
| PMF | 50 | 0.005 | 0.05 | 30 | 5 | 1.102220 |
| PMF | 30 | 0.005 | 0.05 | 20 | 4 | 1.118252 |
| PMF | 20 | 0.005 | 0.05 | 15 | 4 | 1.137063 |
| PMF | 50 | 0.005 | 0.10 | 30 | 5 | 1.137090 |

La mejor configuración obtenida fue `n_factors=50`, `lr=0.005`, `reg=0.05`, `epochs=30` y `patience=5`, alcanzando un RMSE en test de **1.1022**. Al aumentar la regularización a `reg=0.10`, el resultado empeoró, lo que indica que el modelo quedó demasiado limitado. También se observa que aumentar el número de factores mejora el rendimiento, ya que el modelo con 50 factores obtiene mejor RMSE que los modelos con 20 y 30 factores.